In [1]:
from langchain_chroma import Chroma
# Query
def query(vector_store: Chroma, question: str, project: str = None, k: int = 5):
    """Search the vector store and print results."""
    search_kwargs = {"k": k}
    if project:
        search_kwargs["filter"] = {"project": project}
 
    results = vector_store.similarity_search(question, **search_kwargs)
 
    print(f"\nQuery: {question}")
    print(f"Results ({len(results)}):\n")
    for i, doc in enumerate(results, 1):
        print(f"  {i}. [{doc.metadata['source']}]")
        print(f"     {doc.metadata.get('heading_trail', '')}")
        print(f"     {doc.page_content[:150]}...")
        print(f"{'-'*50}")
 
    return results


In [2]:
# 
from langchain_openai import OpenAIEmbeddings
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model = embedding_model)
vector_store = Chroma(
    collection_name = "fastapi",
    embedding_function=embeddings,
    persist_directory="../vectorstore"
)
question = "How to use FastAPI"
query(vector_store, question, project="fastapi", k=10 )


Query: How to use FastAPI
Results (10):

  1. [index.md]
     FastAPI { #fastapi }
     FastAPI { #fastapi }

# FastAPI { #fastapi }  
FastAPI framework, high performance, easy to learn, fast to code, ready for production  
---  
**Docume...
--------------------------------------------------
  2. [index.md]
     Tutorial - User Guide { #tutorial-user-guide } > Install FastAPI { #install-fastapi }
     Tutorial - User Guide { #tutorial-user-guide } > Install FastAPI { #install-fastapi }

## Install FastAPI { #install-fastapi }  
The first step is to ...
--------------------------------------------------
  3. [index.md]
     FastAPI { #fastapi }
     FastAPI { #fastapi }

The key features are:  
* **Fast**: Very high performance, on par with **NodeJS** and **Go** (thanks to Starlette and Pydantic)....
--------------------------------------------------
  4. [first-steps.md]
     First Steps { #first-steps } > Recap, step by step { #recap-step-by-step } > Step 2: create a `FastAPI` "insta

[Document(id='9fbb0e2c29a0e59a77576c6fb67b9fd8', metadata={'source': 'index.md', 'project': 'fastapi', 'heading_trail': 'FastAPI { #fastapi }'}, page_content='FastAPI { #fastapi }\n\n# FastAPI { #fastapi }  \nFastAPI framework, high performance, easy to learn, fast to code, ready for production  \n---  \n**Documentation**: [https://fastapi.tiangolo.com](https://fastapi.tiangolo.com)  \n**Source Code**: [https://github.com/fastapi/fastapi](https://github.com/fastapi/fastapi)  \n---  \nFastAPI is a modern, fast (high-performance), web framework for building APIs with Python based on standard Python type hints.  \nThe key features are:'),
 Document(id='d4216d963ed4148676f4186d44d6bbec', metadata={'source': 'index.md', 'heading_trail': 'Tutorial - User Guide { #tutorial-user-guide } > Install FastAPI { #install-fastapi }', 'project': 'fastapi'}, page_content='Tutorial - User Guide { #tutorial-user-guide } > Install FastAPI { #install-fastapi }\n\n## Install FastAPI { #install-fastapi }  \n

In [3]:
# Check if ChromaDB has any data
import chromadb

client = chromadb.PersistentClient(path="../vectorstore")
collection = client.get_collection("fastapi")
print(f"Total chunks stored: {collection.count()}")

Total chunks stored: 3294


In [4]:
# Eval... 
# dataset

eval_set = [
    {
        "question": "What is FastAPI?",
        "expected_sources": ["index.md"],
        "expected_keywords": ["framework", "python", "api", "high-performance"],
    },
    {
        "question": "How to use OAuth2 with JWT in FastAPI?",
        "expected_sources": ["oauth2-jwt.md"],
        "expected_keywords": ["token", "jwt", "password", "bearer"],
    },
    {
        "question": "How to deploy FastAPI with Docker?",
        "expected_sources": ["docker.md"],
        "expected_keywords": ["docker", "container", "dockerfile"],
    },
    {
        "question": "How to handle path parameters?",
        "expected_sources": ["path-params.md"],
        "expected_keywords": ["path", "parameter", "type"],
    },
    {
        "question": "How to test a FastAPI application?",
        "expected_sources": ["testing.md"],
        "expected_keywords": ["test", "testclient", "pytest"],
    },
]

In [5]:
# Eval... retrieval quality

def evaluate_retrieval(vector_store, eval_set, project="fastapi", k=5):
    results = {
        "total": len(eval_set),
        "source_hits": 0,
        "keyword_hits": 0,
        "details": [],
    }

    for test in eval_set:
        retrieved = vector_store.similarity_search(
            test["question"], k=k, filter={"project": project}
        )

        # Check 1: Did the right source file appear in top-k?
        retrieved_sources = [doc.metadata["source"] for doc in retrieved]
        source_found = any(
            expected in retrieved_sources
            for expected in test["expected_sources"]
        )

        # Check 2: Do retrieved chunks contain expected keywords?
        all_text = " ".join(doc.page_content.lower() for doc in retrieved)
        keywords_found = sum(
            1 for kw in test["expected_keywords"]
            if kw.lower() in all_text
        )
        keyword_score = keywords_found / len(test["expected_keywords"])

        results["source_hits"] += int(source_found)
        results["keyword_hits"] += keyword_score

        results["details"].append({
            "question": test["question"],
            "source_match": source_found,
            "keyword_score": f"{keyword_score:.0%}",
            "top_sources": retrieved_sources[:3],
        })

    # Summary
    source_accuracy = results["source_hits"] / results["total"]
    keyword_accuracy = results["keyword_hits"] / results["total"]

    print(f"\n{'='*50}")
    print(f"  RETRIEVAL EVALUATION")
    print(f"  Source accuracy:  {source_accuracy:.0%}")
    print(f"  Keyword accuracy: {keyword_accuracy:.0%}")
    print(f"{'='*50}\n")

    for d in results["details"]:
        status = "✓" if d["source_match"] else "✗"
        print(f"  {status} {d['question']}")
        print(f"    Keywords: {d['keyword_score']}")
        print(f"    Got: {d['top_sources']}")
        print()

    return results


# Run it
evaluate_retrieval(vector_store, eval_set)


  RETRIEVAL EVALUATION
  Source accuracy:  100%
  Keyword accuracy: 93%

  ✓ What is FastAPI?
    Keywords: 100%
    Got: ['index.md', 'index.md', 'versions.md']

  ✓ How to use OAuth2 with JWT in FastAPI?
    Keywords: 100%
    Got: ['oauth2-jwt.md', 'index.md', 'simple-oauth2.md']

  ✓ How to deploy FastAPI with Docker?
    Keywords: 100%
    Got: ['docker.md', 'docker.md', 'docker.md']

  ✓ How to handle path parameters?
    Keywords: 67%
    Got: ['path-params.md', 'path-params.md', 'path-params.md']

  ✓ How to test a FastAPI application?
    Keywords: 100%
    Got: ['testing.md', 'testing.md', 'index.md']



{'total': 5,
 'source_hits': 5,
 'keyword_hits': 4.666666666666666,
 'details': [{'question': 'What is FastAPI?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['index.md', 'index.md', 'versions.md']},
  {'question': 'How to use OAuth2 with JWT in FastAPI?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['oauth2-jwt.md', 'index.md', 'simple-oauth2.md']},
  {'question': 'How to deploy FastAPI with Docker?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['docker.md', 'docker.md', 'docker.md']},
  {'question': 'How to handle path parameters?',
   'source_match': True,
   'keyword_score': '67%',
   'top_sources': ['path-params.md', 'path-params.md', 'path-params.md']},
  {'question': 'How to test a FastAPI application?',
   'source_match': True,
   'keyword_score': '100%',
   'top_sources': ['testing.md', 'testing.md', 'index.md']}]}

In [6]:


from langchain_chroma import Chroma
from langchain_core.documents import Document


def search_with_threshold(
    vector_store: Chroma,
    query: str,
    project: str | None = None,
    k: int = 10,
    min_similarity: float = 0.20,
    fallback_k: int = 3,
) -> list[Document]:
    """
    Retrieve chunks above a similarity threshold, with a fallback.

    Strategy:
      1. Pull top-k candidates from Chroma (filtered by project if given).
      2. Convert distance -> similarity (1 - distance).
      3. Keep everything >= min_similarity.
      4. If nothing clears the threshold, fall back to top-`fallback_k`
         candidates marked with below_threshold=True so the caller knows.

    Each returned Document carries:
      - metadata["similarity"]      float in roughly [0, 1]
      - metadata["below_threshold"] True only when fallback was used
    """

    # Optional project filter — Chroma's `where` clause, exact-match metadata
    search_kwargs = {"k": k}
    if project:
        search_kwargs["filter"] = {"project": project}

    raw = vector_store.similarity_search_with_score(query, **search_kwargs)

    # Convert distance -> similarity for every candidate
    scored = []
    for doc, distance in raw:
        similarity = 1 - distance
        # Mutate a copy of metadata to avoid surprising other callers
        doc.metadata = {
            **doc.metadata,
            "similarity": similarity,
            "below_threshold": False,
        }
        scored.append((doc, similarity))

    # Sort high-to-low (Chroma usually does this, but make it explicit)
    scored.sort(key=lambda x: x[1], reverse=True)

    # Apply threshold
    passing = [doc for doc, sim in scored if sim >= min_similarity]

    if passing:
        return passing

    # Fallback: nothing cleared the bar — return top few, flagged
    fallback = []
    for doc, sim in scored[:fallback_k]:
        doc.metadata = {**doc.metadata, "below_threshold": True}
        fallback.append(doc)

    return fallback

In [7]:
# Eval... retrieval quality with threshold

def evaluate_threshold_retrieval(
    vector_store, eval_set, project="fastapi", k=10, min_similarity=0.20
):
    results = {
        "total": len(eval_set),
        "source_hits": 0,
        "keyword_hits": 0,
        "avg_similarity": 0,
        "avg_results_returned": 0,
        "fallback_count": 0,
        "details": [],
    }

    all_similarities = []

    for test in eval_set:
        retrieved = search_with_threshold(
            vector_store, test["question"],
            project=project, k=k, min_similarity=min_similarity,
        )
        print(retrieved)

        # Check 1: Source match
        retrieved_sources = [doc.metadata["source"] for doc in retrieved]
        source_found = any(
            expected in retrieved_sources
            for expected in test["expected_sources"]
        )

        # Check 2: Keyword match
        all_text = " ".join(doc.page_content.lower() for doc in retrieved)
        keywords_found = sum(
            1 for kw in test["expected_keywords"]
            if kw.lower() in all_text
        )
        keyword_score = keywords_found / len(test["expected_keywords"])

        # Check 3: Similarity scores
        similarities = [doc.metadata.get("similarity", 0) for doc in retrieved]
        avg_sim = sum(similarities) / len(similarities) if similarities else 0
        all_similarities.extend(similarities)

        # Check 4: Did fallback trigger?
        used_fallback = any(doc.metadata.get("below_threshold") for doc in retrieved)

        results["source_hits"] += int(source_found)
        results["keyword_hits"] += keyword_score
        results["avg_results_returned"] += len(retrieved)
        results["fallback_count"] += int(used_fallback)

        results["details"].append({
            "question": test["question"],
            "source_match": source_found,
            "keyword_score": f"{keyword_score:.0%}",
            "results_returned": len(retrieved),
            "similarities": [f"{s:.4f}" for s in similarities],
            "avg_similarity": f"{avg_sim:.4f}",
            "used_fallback": used_fallback,
            "top_sources": retrieved_sources[:3],
        })

    # Summary
    total = results["total"]
    source_accuracy = results["source_hits"] / total
    keyword_accuracy = results["keyword_hits"] / total
    avg_returned = results["avg_results_returned"] / total
    avg_sim = sum(all_similarities) / len(all_similarities) if all_similarities else 0

    print(f"\n{'='*55}")
    print(f"  THRESHOLD RETRIEVAL EVALUATION (min={min_similarity})")
    print(f"{'='*55}")
    print(f"  Source accuracy:     {source_accuracy:.0%}")
    print(f"  Keyword accuracy:    {keyword_accuracy:.0%}")
    print(f"  Avg similarity:      {avg_sim:.4f}")
    print(f"  Avg results/query:   {avg_returned:.1f} / {k}")
    print(f"  Fallback triggered:  {results['fallback_count']} / {total}")
    print(f"{'='*55}\n")

    for d in results["details"]:
        status = "✓" if d["source_match"] else "✗"
        fallback = " (FALLBACK)" if d["used_fallback"] else ""
        print(f"  {status} {d['question']}{fallback}")
        print(f"    Keywords:    {d['keyword_score']}")
        print(f"    Returned:    {d['results_returned']} chunks")
        print(f"    Avg sim:     {d['avg_similarity']}")
        print(f"    Scores:      {d['similarities']}")
        print(f"    Sources:     {d['top_sources']}")
        print()

    return results


# Run at different thresholds to find the sweet spot
for threshold in [0.10, 0.15, 0.20]:
    evaluate_threshold_retrieval(
        vector_store, eval_set, min_similarity=threshold
    )

[Document(id='b252af4913f418ca25fa985234ddd183', metadata={'source': 'index.md', 'heading_trail': 'FastAPI { #fastapi }', 'project': 'fastapi', 'similarity': 0.7319144606590271, 'below_threshold': False}, page_content='FastAPI { #fastapi }\n\nThe key features are:  \n* **Fast**: Very high performance, on par with **NodeJS** and **Go** (thanks to Starlette and Pydantic). [One of the fastest Python frameworks available](#performance).\n* **Fast to code**: Increase the speed to develop features by about 200% to 300%. *\n* **Fewer bugs**: Reduce about 40% of human (developer) induced errors. *\n* **Intuitive**: Great editor support. Completion everywhere. Less time debugging.'), Document(id='9fbb0e2c29a0e59a77576c6fb67b9fd8', metadata={'source': 'index.md', 'project': 'fastapi', 'heading_trail': 'FastAPI { #fastapi }', 'similarity': 0.7173213362693787, 'below_threshold': False}, page_content='FastAPI { #fastapi }\n\n# FastAPI { #fastapi }  \nFastAPI framework, high performance, easy to lea

In [8]:
# Debug: see what ChromaDB is actually returning
results_with_scores = vector_store.similarity_search_with_score(
    "What is FastAPI?", k=5, filter={"project": "fastapi"}
)

for doc, score in results_with_scores:
    print(f"  Raw score: {score:.6f}  |  1-score: {1-score:.6f}  |  {doc.metadata['source']}")

  Raw score: 0.268086  |  1-score: 0.731914  |  index.md
  Raw score: 0.282679  |  1-score: 0.717321  |  index.md
  Raw score: 0.344028  |  1-score: 0.655972  |  versions.md
  Raw score: 0.358664  |  1-score: 0.641336  |  fastapicloud.md
  Raw score: 0.366023  |  1-score: 0.633977  |  fastapi-cli.md


In [9]:
eval_set = [
    # ─────────────────────────────────────────────────────────
    # TIER 1 — Easy: query phrasing closely matches doc titles
    # These should all hit rank 1 reliably.
    # ─────────────────────────────────────────────────────────
    {
        "question": "What is FastAPI?",
        "expected_sources": ["index.md", "features.md"],
        "expected_keywords": ["framework", "python", "api", "high-performance"],
    },
    {
        "question": "How to handle path parameters?",
        "expected_sources": ["path-params.md"],
        "expected_keywords": ["path", "parameter", "type"],
    },
    {
        "question": "How to use query parameters?",
        "expected_sources": ["query-params.md"],
        "expected_keywords": ["query", "parameter", "default"],
    },
    {
        "question": "How to receive a request body in FastAPI?",
        "expected_sources": ["body.md"],
        "expected_keywords": ["body", "pydantic", "model", "json"],
    },
    {
        "question": "How to deploy FastAPI with Docker?",
        "expected_sources": ["docker.md"],
        "expected_keywords": ["docker", "container", "dockerfile"],
    },
    {
        "question": "How to test a FastAPI application?",
        "expected_sources": ["testing.md"],
        "expected_keywords": ["test", "testclient", "pytest"],
    },
    {
        "question": "How to use OAuth2 with JWT in FastAPI?",
        "expected_sources": ["oauth2-jwt.md"],
        "expected_keywords": ["token", "jwt", "password", "bearer"],
    },
    {
        "question": "How to add CORS middleware?",
        "expected_sources": ["cors.md"],
        "expected_keywords": ["cors", "origin", "middleware"],
    },
    {
        "question": "How to upload files in FastAPI?",
        "expected_sources": ["request-files.md"],
        "expected_keywords": ["file", "upload", "uploadfile"],
    },
    {
        "question": "How to serve static files?",
        "expected_sources": ["static-files.md"],
        "expected_keywords": ["static", "files", "mount"],
    },

    # ─────────────────────────────────────────────────────────
    # TIER 2 — Medium: query phrasing differs from doc titles,
    # may overlap with adjacent topics
    # ─────────────────────────────────────────────────────────
    {
        "question": "How do I validate string lengths on query parameters?",
        "expected_sources": ["query-params-str-validations.md"],
        "expected_keywords": ["query", "validation", "max_length", "min_length"],
    },
    {
        "question": "How to add numeric constraints to path parameters?",
        "expected_sources": ["path-params-numeric-validations.md"],
        "expected_keywords": ["path", "numeric", "ge", "le", "gt"],
    },
    {
        "question": "How to use Pydantic Field for body validation?",
        "expected_sources": ["body-fields.md"],
        "expected_keywords": ["field", "pydantic", "validation"],
    },
    {
        "question": "How do I use nested Pydantic models in request bodies?",
        "expected_sources": ["body-nested-models.md"],
        "expected_keywords": ["nested", "model", "list", "pydantic"],
    },
    {
        "question": "How do I read cookies from a request?",
        "expected_sources": ["cookie-params.md"],
        "expected_keywords": ["cookie", "parameter"],
    },
    {
        "question": "How to read custom request headers?",
        "expected_sources": ["header-params.md"],
        "expected_keywords": ["header", "parameter"],
    },
    {
        "question": "How to handle form data submissions?",
        "expected_sources": ["request-forms.md"],
        "expected_keywords": ["form", "data"],
    },
    {
        "question": "How to return errors with status codes?",
        "expected_sources": ["handling-errors.md"],
        "expected_keywords": ["error", "httpexception", "status"],
    },
    {
        "question": "How to declare what my endpoint returns?",
        "expected_sources": ["response-model.md"],
        "expected_keywords": ["response", "model", "return"],
    },
    {
        "question": "How to change the HTTP status code of a response?",
        "expected_sources": ["response-status-code.md"],
        "expected_keywords": ["status", "code", "response"],
    },
    {
        "question": "How does dependency injection work in FastAPI?",
        "expected_sources": ["dependencies.md", "index.md"],  # may live in dependencies/index.md
        "expected_keywords": ["depends", "injection", "dependency"],
    },
    {
        "question": "How to run code in the background after returning a response?",
        "expected_sources": ["background-tasks.md"],
        "expected_keywords": ["background", "tasks", "async"],
    },
    {
        "question": "How to organize a large FastAPI project with multiple files?",
        "expected_sources": ["bigger-applications.md"],
        "expected_keywords": ["router", "apirouter", "include"],
    },
    {
        "question": "How to use a database with FastAPI?",
        "expected_sources": ["sql-databases.md"],
        "expected_keywords": ["sql", "database", "sqlmodel"],
    },
    {
        "question": "How to add custom middleware to my app?",
        "expected_sources": ["middleware.md"],
        "expected_keywords": ["middleware", "request", "response"],
    },

    # ─────────────────────────────────────────────────────────
    # TIER 3 — Harder: conceptual / paraphrased / multi-topic
    # These exercise breadcrumb embedding and re-ranker need
    # ─────────────────────────────────────────────────────────
    {
        "question": "When should I use async def vs def for route handlers?",
        "expected_sources": ["async.md"],
        "expected_keywords": ["async", "await", "concurrency"],
    },
    {
        "question": "Why does FastAPI use Python type hints?",
        "expected_sources": ["python-types.md", "features.md"],
        "expected_keywords": ["type", "hints", "annotations"],
    },
    {
        "question": "How is FastAPI different from Flask or Django?",
        "expected_sources": ["alternatives.md"],
        "expected_keywords": ["flask", "django", "comparison"],
    },
    {
        "question": "How do I get the currently logged-in user in an endpoint?",
        "expected_sources": ["get-current-user.md", "simple-oauth2.md", "oauth2-jwt.md"],
        "expected_keywords": ["current", "user", "depends", "token"],
    },
    {
        "question": "How to convert Pydantic models to JSON-compatible dicts?",
        "expected_sources": ["encoder.md"],
        "expected_keywords": ["jsonable_encoder", "json", "pydantic"],
    },
]

In [10]:
# MRR Eval
def evaluate_retrieval(eval_set, vectorstore, k=10, threshold=0.10):
    """
    Evaluate retrieval quality with rank-aware metrics + keyword check.

    Metrics:
    - source_found:   did any expected source appear in top-k? (recall floor)
    - hit_at_1:       was the very top result correct?         (strict)
    - hit_at_3:       was a correct result in top 3?           (matches typical LLM context)
    - mrr:            Mean Reciprocal Rank — rewards high ranking
    - keyword_acc:    fraction of expected keywords found across retrieved chunks
    - avg_top_score:  average similarity of rank-1 result      (sanity check)
    """

    def source_matches(retrieved, expected_set):
        """Tolerant match: 'index.md' matches 'fastapi/index.md' and vice versa."""
        return any(
            retrieved.endswith(exp) or exp.endswith(retrieved)
            for exp in expected_set
        )

    totals = {
        "source_found": 0,
        "hit_at_1": 0,
        "hit_at_3": 0,
        "mrr": 0.0,
        "keyword_acc": 0.0,
        "avg_top_score": 0.0,
    }
    per_query = []

    for test in eval_set:
        query = test["question"]
        expected_sources = set(test["expected_sources"])
        expected_keywords = [kw.lower() for kw in test.get("expected_keywords", [])]

        # Retrieve with scores, apply loose threshold as garbage filter
        raw = vectorstore.similarity_search_with_score(query, k=k)
        filtered = [(doc, 1 - dist) for doc, dist in raw if (1 - dist) >= threshold]

        sources = [doc.metadata["source"] for doc, _ in filtered]
        top_score = filtered[0][1] if filtered else 0.0

        # Combined text from retrieved chunks for keyword check
        retrieved_text = " ".join(doc.page_content for doc, _ in filtered).lower()

        # source_found — anywhere in top-k
        found = any(source_matches(s, expected_sources) for s in sources)

        # hit@1 — top result correct
        hit1 = bool(sources) and source_matches(sources[0], expected_sources)

        # hit@3 — anything correct in top 3
        hit3 = any(source_matches(s, expected_sources) for s in sources[:3])

        # MRR — reciprocal of rank of first correct hit
        rank = next(
            (i + 1 for i, s in enumerate(sources) if source_matches(s, expected_sources)),
            None,
        )
        rr = 1 / rank if rank else 0.0

        # Keyword accuracy — fraction of expected keywords present in retrieved text
        if expected_keywords:
            kw_hits = sum(1 for kw in expected_keywords if kw in retrieved_text)
            kw_acc = kw_hits / len(expected_keywords)
        else:
            kw_acc = 0.0

        totals["source_found"] += int(found)
        totals["hit_at_1"] += int(hit1)
        totals["hit_at_3"] += int(hit3)
        totals["mrr"] += rr
        totals["keyword_acc"] += kw_acc
        totals["avg_top_score"] += top_score

        per_query.append({
            "question": query,
            "expected_sources": list(expected_sources),
            "retrieved_top3": sources[:3],
            "first_correct_rank": rank,
            "reciprocal_rank": round(rr, 3),
            "keyword_acc": round(kw_acc, 3),
            "top_score": round(top_score, 3),
        })

    n = len(eval_set)
    summary = {key: round(val / n, 3) for key, val in totals.items()}

    return {"summary": summary, "per_query": per_query}


def print_eval_report(results):
    """Pretty-print eval results, surface worst queries for debugging."""
    s = results["summary"]
    print("=" * 60)
    print("RETRIEVAL EVALUATION")
    print("=" * 60)
    print(f"  Source found (any rank):  {s['source_found']:.0%}")
    print(f"  Hit@1 (top result right): {s['hit_at_1']:.0%}")
    print(f"  Hit@3 (right in top 3):   {s['hit_at_3']:.0%}")
    print(f"  MRR (rank quality):       {s['mrr']:.3f}")
    print(f"  Keyword accuracy:         {s['keyword_acc']:.0%}")
    print(f"  Avg top-1 score:          {s['avg_top_score']:.3f}")
    print()

    # Sort worst queries first so you immediately see what to debug
    worst = sorted(results["per_query"], key=lambda x: x["reciprocal_rank"])
    print("PER-QUERY BREAKDOWN (worst first):")
    print("-" * 60)
    for q in worst:
        rank_str = q["first_correct_rank"] if q["first_correct_rank"] else "NOT FOUND"
        print(f"  Q: {q['question']}")
        print(f"     First correct rank: {rank_str}  |  RR: {q['reciprocal_rank']}  |  Top score: {q['top_score']}")
        print(f"     Keyword acc: {q['keyword_acc']:.0%}")
        print(f"     Expected: {q['expected_sources']}")
        print(f"     Got top-3: {q['retrieved_top3']}")
        print()

In [17]:
test_set = [
    {
        "query": "What is FastAPI?",
        "expected_sources": ["fastapi/index.md", "fastapi/features.md"],
        # keep your existing "expected_keywords" if you also want keyword eval
    },
    # ... more test cases
]

results = evaluate_retrieval(eval_set, vector_store, k=10, threshold=0.3)
print_eval_report(results)

RETRIEVAL EVALUATION
  Source found (any rank):  100%
  Hit@1 (top result right): 87%
  Hit@3 (right in top 3):   93%
  MRR (rank quality):       0.907
  Keyword accuracy:         100%
  Avg top-1 score:          0.624

PER-QUERY BREAKDOWN (worst first):
------------------------------------------------------------
  Q: How to read custom request headers?
     First correct rank: 6  |  RR: 0.167  |  Top score: 0.651
     Keyword acc: 100%
     Expected: ['header-params.md']
     Got top-3: ['response-headers.md', 'response-headers.md', 'handling-errors.md']

  Q: How is FastAPI different from Flask or Django?
     First correct rank: 5  |  RR: 0.2  |  Top score: 0.686
     Keyword acc: 100%
     Expected: ['alternatives.md']
     Got top-3: ['index.md', 'index.md', 'versions.md']

  Q: How do I read cookies from a request?
     First correct rank: 3  |  RR: 0.333  |  Top score: 0.485
     Keyword acc: 100%
     Expected: ['cookie-params.md']
     Got top-3: ['response-cookies.md', 'cook

In [12]:
all_data = vector_store._collection.get()
sources_present = sorted(set(m["source"] for m in all_data["metadatas"]))

failed_files = [
    "cookie-params.md",
    "request-forms.md",
    "response-model.md",
    "get-current-user.md",
]

print("Files in corpus:", len(sources_present))
print()
for f in failed_files:
    status = "✓ IN CORPUS" if f in sources_present else "✗ MISSING"
    print(f"  {status}: {f}")

print("\nAll sources containing 'cookie', 'form', 'response-model', or 'current-user':")
for s in sources_present:
    if any(kw in s.lower() for kw in ["cookie", "form", "response-model", "current-user", "current_user"]):
        print(f"  {s}")

Files in corpus: 136

  ✓ IN CORPUS: cookie-params.md
  ✓ IN CORPUS: request-forms.md
  ✓ IN CORPUS: response-model.md
  ✓ IN CORPUS: get-current-user.md

All sources containing 'cookie', 'form', 'response-model', or 'current-user':
  cookie-param-models.md
  cookie-params.md
  get-current-user.md
  request-form-models.md
  request-forms-and-files.md
  request-forms.md
  response-cookies.md
  response-model.md


In [13]:
# Direct call to Chroma, no threshold filtering at all
query = "How do I read cookies from a request?"
raw = vector_store.similarity_search_with_score(query, k=10)

print(f"Raw results returned: {len(raw)}")
print()
for doc, dist in raw:
    sim = 1 - dist
    print(f"  dist={dist:.4f}  sim={sim:+.4f}  source={doc.metadata['source']}")

Raw results returned: 10

  dist=0.5146  sim=+0.4854  source=response-cookies.md
  dist=0.5447  sim=+0.4553  source=cookie-param-models.md
  dist=0.5465  sim=+0.4535  source=cookie-params.md
  dist=0.5511  sim=+0.4489  source=cookie-params.md
  dist=0.5573  sim=+0.4427  source=response-cookies.md
  dist=0.5621  sim=+0.4379  source=response-cookies.md
  dist=0.5627  sim=+0.4373  source=cookie-param-models.md
  dist=0.5715  sim=+0.4285  source=cookie-params.md
  dist=0.5777  sim=+0.4223  source=cookie-param-models.md
  dist=0.5808  sim=+0.4192  source=cookie-param-models.md
